# cv_r1 学習フェーズ【Colab・要 GPU】

setup（環境構築）→ deploy（配置・検証ゲート）の**後**に実行する3本目。bert_gen → style_gen → 学習を
このノート1本で回し、**実行結果（ログ）をセルに残す**。トラブル時はこのノートの出力を見れば切り分けできる。

**前提（未達なら止まる）**
- setup 実行済み（fork clone・initialize 済み。Drive に永続）
- deploy §3 検証を**全通過**（`Data/cv_r1/config.json` と esd/wav/cadseq が配置済み）
- **GPU ランタイム**（[ランタイム]→[ランタイムのタイプを変更]→GPU）

**このノートでやらないこと**
- `preprocess_text` は**走らせない**（esd 配置済み・spk2id を再生成し得る＝emb_g 行ズレの原因）


In [ ]:
# ===== §1 マウント・cwd・前提チェック =====
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, subprocess

BASE = Path("/content/drive/MyDrive/Style-Bert-VITS2")   # setup/deploy と同じ値
assert BASE.exists(), f"BASE が無い: {BASE}（先に setup を実行）"
os.chdir(BASE); print("cwd:", Path.cwd())

br = subprocess.run(["git","rev-parse","--abbrev-ref","HEAD"], capture_output=True, text=True).stdout.strip()
assert br == "layer-b-cadence-seq", f"★branch が違う: {br}"
print("branch:", br)

# deploy ゲート通過の確認
DATA = BASE / "Data" / "cv_r1"
for f in ["config.json","esd_train.list","esd_val.list"]:
    assert (DATA/f).exists(), f"★{DATA/f} が無い → 先に deploy を全通過させる"
print("Data/cv_r1: config.json / esd 配置 OK")

# GPU ランタイム確認（torch を import せず nvidia-smi で。§2 の torch 入替と干渉させない）
g = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
assert g.returncode == 0 and "GPU" in g.stdout, "★GPU ランタイムでない → [ランタイム]→[ランタイムのタイプを変更]→GPU"
print(g.stdout.strip())


In [ ]:
# ===== §2 依存インストール（この GPU セッションで実行）=====
# pip はセッション揮発なので、setup を実行した CPU セッションとは別のこの GPU セッションで再度必要。
# setup §2 と同一：faster-whisper/av を除外して av のソースビルド失敗を回避（07a 方式）。
# torch は requirements の pin(2.3.1/cu121)のまま＝標準 Colab GPU(T4/L4/A100)で動作。restart も shim も不要。
import subprocess, sys, re
from pathlib import Path
req = Path("requirements.txt"); req_train = Path("requirements_no_whisper.txt")
_drop = re.compile(r"^(faster-whisper|av)==")
kept = [l for l in req.read_text(encoding="utf-8").splitlines() if not _drop.match(l.strip())]
req_train.write_text("\n".join(kept) + "\n", encoding="utf-8")
r = subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(req_train)], capture_output=True, text=True)
print(r.stdout[-1500:] or "(quiet)")
if r.returncode != 0:
    print(r.stderr[-4000:]); raise SystemExit("pip 失敗")
# torch は subprocess で確認（カーネル再起動不要にするため in-kernel import しない）
subprocess.run([sys.executable,"-c","import torch;print('torch',torch.__version__,'cuda',torch.cuda.is_available())"])


## §3〜§5 前処理と学習

以降は `!python <script>` 形式で実行し、進捗ログをセルに残す（`subprocess` で握りつぶさない）。
すべて cwd = BASE（fork 直下）で走り、`Data/cv_r1/config.json` を読む。


In [ ]:
# ===== §3 bert_gen（DeBERTa 特徴量 .bert.pt を各発話に生成）=====
# config の training_files/validation_files（=esd）を読み、各 wav の隣に {wav}.bert.pt を書く。数分〜。
!python bert_gen.py -c Data/cv_r1/config.json
# 生成数の目安確認（非致命）
import subprocess
n = subprocess.run("find Data/cv_r1 -name '*.bert.pt' | wc -l", shell=True, capture_output=True, text=True).stdout.strip()
print("生成された .bert.pt:", n, "（esd train 14226 + val 3789 = 18015 が目安）")


In [ ]:
# ===== §4 style_gen（スタイルベクトル生成。num_styles=1 / Neutral のみ）=====
# ※ preprocess_text は走らせない（spk2id 再生成の恐れ）。style_gen は esd/config から直接生成。
!python style_gen.py -c Data/cv_r1/config.json
import os
sv = "Data/cv_r1/style_vectors.npy"
print("style_vectors.npy:", "OK" if os.path.exists(sv) else "★未生成（style_gen の出力を確認）")


In [ ]:
# ===== §5 学習 =====
# -m は「データセットフォルダのパス」= Data/cv_r1（★"cv_r1" 単体は誤り。model_dir がこの下に連結される）。
#   → checkpoint は Data/cv_r1/models/ に保存される。
# batch=16 / 10 epoch ≈ 8,900 step, freeze_decoder=True は config に設定済み（底モデル=pretrained_jp_extra/{G,D,WD}_0）。
# 途中切断したら、この同じセルを再実行すれば models/ の最新 checkpoint から再開する（SBV2 の挙動）。
!python train_ms_jp_extra.py -c Data/cv_r1/config.json -m Data/cv_r1


## 出力・再開・トラブル対応

- **checkpoint**: `Data/cv_r1/models/`（`G_*.pth` / `D_*.pth` / `WD_*.pth`）。Drive に永続。
- **再開**: Colab 切断後は §1→§2 を実行してから §5 を再実行（最新 checkpoint から再開）。§3/§4 は生成物が Drive に残るので再実行不要。
- **ログ**: `Data/cv_r1/train_*.log` にも残る。
- **GPU が出ない**: [ランタイム]→[ランタイムのタイプを変更]→GPU。
- **§2 で失敗**: `requirements_no_whisper.txt` に `stable_ts` 等が残ってビルドで詰まる場合、§2 の除外規則にそのパッケージ名を足す（学習には不要）。
- **Blackwell(sm_120)機で学習する場合のみ**: torch を 2.11+cu128 に入替＋soundfile shim が必要（07a §2/§3 参照）。標準 Colab GPU では不要。
